<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-05/AI_Modul_5.6_Praktikum_Proses_Training_Model.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 5.6: Praktikum Komputasi - Proses Training Model dalam Machine Learning
Program Studi Sarjana Sains Data / Kecerdasan Buatan / Teknik Pertanian  
Fakultas Teknologi Pertanian, Institut Pertanian STIPER Yogyakarta

---

### Tujuan Praktikum:
1. Memahami perumusan fungsi biaya matematis $J(\boldsymbol{\theta})$ dan vektor gradien $\nabla J(\boldsymbol{\theta})$.
2. Mengimplementasikan secara murni (*from scratch*) tiga varian algoritma optimasi: *Batch GD*, *Stochastic GD*, dan *Mini-Batch GD*.
3. Menganalisis pengaruh pemilihan tingkat pembelajaran (*learning rate* $\eta$) terhadap konvergensi numerik.
4. Membuktikan peran krusial standarisasi fitur dalam mencegah distorsi kurvatur lanskap fungsi biaya.
5. Memanfaatkan estimator *Scikit-Learn* berbasis SGD (`SGDRegressor`, `SGDClassifier`) untuk pelatihan data skala besar dengan *Early Stopping*.

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import SGDRegressor, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150
print("Environment komputasi praktikum berhasil diinisialisasi.")


## 1. Pembangkitan Data Sintetis Irigasi & Normalisasi Fitur

Data sintetis memodelkan kebutuhan volume air irigasi harian kelapa sawit ($liter/pohon/hari$) berdasarkan dua variabel lingkungan:
- $X_1$: Radiasi Matahari ($MJ/m^2/hari$, rentang 10 - 25)
- $X_2$: Defisit Tekanan Uap Air / VPD ($kPa$, rentang 0.5 - 3.5)


In [ ]:
np.random.seed(42)
n_samples = 600

radiasi_surya = np.random.uniform(10.0, 25.0, n_samples)
vpd_uap = np.random.uniform(0.5, 3.5, n_samples)

# Matriks Fitur Mentah
X_raw = np.column_stack([radiasi_surya, vpd_uap])

# Target Sejati: y = 20.0 + 4.5*X1 + 12.0*X2 + noise
theta_true = np.array([20.0, 4.5, 12.0])
y_true_clean = theta_true[0] + theta_true[1] * radiasi_surya + theta_true[2] * vpd_uap
y = y_true_clean + np.random.normal(0, 3.0, n_samples)

# Normalisasi Z-Score untuk Fitur
scaler = StandardScaler()
X_norm = scaler.fit_transform(X_raw)

# Tambahkan kolom intercept (x0 = 1)
X_b = np.hstack([np.ones((n_samples, 1)), X_norm])

print(f"Bentuk Matriks Desain X_b: {X_b.shape}")
print(f"Rata-rata Target Kebutuhan Air: {np.mean(y):.2f} liter/hari")


## 2. Implementasi Murni Tiga Varian Gradient Descent (*From Scratch*)

1. **Batch Gradient Descent (BGD)**: Gradien dihitung atas seluruh $n$ sampel per iterasi.
2. **Stochastic Gradient Descent (SGD)**: Gradien dihitung per 1 sampel tunggal.
3. **Mini-Batch Gradient Descent (MBGD)**: Gradien dihitung per subhimpunan ukuran $b = 32$.


In [ ]:
def run_bgd(X, y, lr=0.1, n_epochs=50):
    n, d = X.shape
    theta = np.zeros(d)
    loss_history = []
    
    for epoch in range(n_epochs):
        # Gradien atas seluruh data
        grad = (2.0 / n) * X.T @ (X @ theta - y)
        theta -= lr * grad
        loss = np.mean((X @ theta - y)**2)
        loss_history.append(loss)
    return theta, loss_history

def run_sgd(X, y, lr=0.01, n_epochs=50):
    n, d = X.shape
    theta = np.zeros(d)
    loss_history = []
    
    for epoch in range(n_epochs):
        indices = np.random.permutation(n)
        for i in indices:
            xi = X[i:i+1]
            yi = y[i:i+1]
            grad = 2.0 * xi.T @ (xi @ theta - yi)
            theta -= lr * grad
        loss = np.mean((X @ theta - y)**2)
        loss_history.append(loss)
    return theta, loss_history

def run_mbgd(X, y, batch_size=32, lr=0.05, n_epochs=50):
    n, d = X.shape
    theta = np.zeros(d)
    loss_history = []
    
    for epoch in range(n_epochs):
        indices = np.random.permutation(n)
        X_shuff = X[indices]
        y_shuff = y[indices]
        for i in range(0, n, batch_size):
            xb = X_shuff[i:i+batch_size]
            yb = y_shuff[i:i+batch_size]
            grad = (2.0 / len(yb)) * xb.T @ (xb @ theta - yb)
            theta -= lr * grad
        loss = np.mean((X @ theta - y)**2)
        loss_history.append(loss)
    return theta, loss_history

# Eksekusi Ketiga Algoritma
theta_bgd, loss_bgd = run_bgd(X_b, y, lr=0.1, n_epochs=40)
theta_sgd, loss_sgd = run_sgd(X_b, y, lr=0.01, n_epochs=40)
theta_mbgd, loss_mbgd = run_mbgd(X_b, y, batch_size=32, lr=0.05, n_epochs=40)

# Solusi Analitik Persamaan Normal OLS
theta_ols = np.linalg.inv(X_b.T @ X_b) @ X_b.T @ y

print(f"Solusi Analitik OLS : {np.round(theta_ols, 3)}")
print(f"Solusi Batch GD     : {np.round(theta_bgd, 3)}")
print(f"Solusi Stochastic GD: {np.round(theta_sgd, 3)}")
print(f"Solusi Mini-Batch GD: {np.round(theta_mbgd, 3)}")


## 3. Analisis Trajektori Konvergensi (*Loss Curves*)

Visualisasi perbandingan laju penurunan nilai fungsi kerugian terhadap jumlah siklus (*epochs*).


In [ ]:
plt.figure(figsize=(9, 4.5))
epochs_range = range(1, 41)

plt.plot(epochs_range, loss_bgd, color='#2980b9', linewidth=2.2, label='Batch GD (Stabil, n=600)')
plt.plot(epochs_range, loss_sgd, color='#e74c3c', linewidth=1.8, linestyle=':', label='Stochastic GD (b=1)')
plt.plot(epochs_range, loss_mbgd, color='#27ae60', linewidth=2.4, linestyle='--', label='Mini-Batch GD (b=32)')

plt.title('Kurva Penurunan Fungsi Biaya (MSE) per Epoch pada Data Irigasi Sawit', fontweight='bold')
plt.xlabel('Siklus Pelatihan (Epochs)', fontweight='bold')
plt.ylabel('Fungsi Biaya J(θ) - MSE', fontweight='bold')
plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.savefig('docs/assets/praktikum_perbandingan_loss_curves.png', bbox_inches='tight')
plt.close()
print("Grafik kurva konvergensi berhasil disimpan.")


## 4. Eksperimen Pengaruh Tingkat Pembelajaran (*Learning Rate* $\eta$)

Menguji perilaku algoritma Mini-Batch GD ketika $\eta$ disetel terlalu kecil (0.001), optimal (0.05), dan terlalu besar (0.95).


In [ ]:
lr_uji = [0.001, 0.05, 0.95]
hist_lr = {}

for lr_val in lr_uji:
    _, loss_hist = run_mbgd(X_b, y, batch_size=32, lr=lr_val, n_epochs=35)
    hist_lr[lr_val] = loss_hist

plt.figure(figsize=(9, 4.5))
for lr_val, col, ls in zip(lr_uji, ['#7f8c8d', '#27ae60', '#c0392b'], [':', '-', '--']):
    plt.plot(range(1, 36), hist_lr[lr_val], color=col, linestyle=ls, linewidth=2.2, label=f'LR = {lr_val}')

plt.title('Dinamika Tingkat Pembelajaran terhadap Kecepatan Konvergensi', fontweight='bold')
plt.xlabel('Epochs', fontweight='bold')
plt.ylabel('Fungsi Biaya MSE (Skala Log)', fontweight='bold')
plt.yscale('log')
plt.legend()
plt.tight_layout()
plt.savefig('docs/assets/praktikum_efek_learning_rate.png', bbox_inches='tight')
plt.close()
print("Eksperimen laju pembelajaran selesai.")


## 5. Implementasi Scikit-Learn `SGDRegressor` dengan Early Stopping

Untuk dataset besar di industri, kita memanfaatkan `sklearn.linear_model.SGDRegressor` yang dilengkapi dengan kriteria berhenti otomatis (`early_stopping=True`).


In [ ]:
sgd_reg = SGDRegressor(
    loss='squared_error',
    penalty='l2',
    alpha=0.001,
    learning_rate='adaptive',
    eta0=0.05,
    max_iter=1000,
    tol=1e-3,
    early_stopping=True,
    n_iter_no_change=5,
    random_state=42
)

sgd_reg.fit(X_norm, y)

y_pred_sgd = sgd_reg.predict(X_norm)
mse_final = mean_squared_error(y, y_pred_sgd)
r2_final = r2_score(y, y_pred_sgd)

print(f"Jumlah Epoch Aktual hingga Konvergen: {sgd_reg.n_iter_}")
print(f"Koefisien Estimasi [w1, w2]        : {np.round(sgd_reg.coef_, 3)}")
print(f"Bias / Intercept (w0)              : {sgd_reg.intercept_[0]:.3f}")
print(f"MSE Evaluasi Akhir                 : {mse_final:.4f}")
print(f"Nilai R2 Score                     : {r2_final:.4f}")


## 6. Latihan Terpandu Mandiri (Tugas Mahasiswa)

1. **Uji Tanpa Standarisasi**: Jalankan fungsi `run_bgd` pada matriks data mentah `X_raw` (tanpa `StandardScaler`) dengan `lr=0.05`. Amati nilai gradien dan jelaskan mengapa algoritma mengalami *overflow* atau osilasi ekstrem.
2. **Implementasi Huber Loss**: Modifikasi fungsi `run_mbgd` agar menghitung gradien berdasarkan *Huber Loss* ($\delta = 2.0$) alih-alih MSE murni. Uji ketahanannya terhadap 10 sampel pencilan (*outliers*) ekstrem buatan!
3. **Simulasi Momentum Polyak**: Tambahkan suku momentum $v_t = eta v_{t-1} + \eta 
abla J(	heta)$ pada aturan pembaruan parameter ($	heta \leftarrow 	heta - v_t$) dengan $eta = 0.9$. Bandingkan kecepatan konvergensi terhadap BGD standar.
